In [58]:
import pandas as pd
import numpy as np
from EssSimulation_withoutMaxDemand import EssSimulationModel
import calendar
import copy
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif']=['SimHei']    # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号

In [59]:
exp_name = "estimate830"
node_name = "route_B_09"

In [60]:
# es_info = {"transform_capacity": 63000,
#            "invertband": 0,
#            "soc_redundant_ratio": 0,
#            "usable_depth": 0.9,
#            "charge_loss": 0.92,
#            "discharge_loss": 0.95,
#            "es_charge_max": 8920,
#            "es_charge_min": -8920,
#            "es_capacity_max": 17888,
#            "es_capacity_min": 0}
es_info = {"transform_capacity": 63000,
           "invertband": 0,
           "soc_redundant_ratio": 0,
           "usable_depth": 0.9,
           "charge_loss": 0.92,
           "discharge_loss": 0.95,
           "es_charge_max": 9000,
           "es_charge_min": -9000,
           "es_capacity_max": 18000,
           "es_capacity_min": 0}

In [61]:
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/schedule_result_evencharge_new1.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

In [62]:
simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 4050)
origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, 38.4)

In [64]:
ori_max_demand = total_load_df["total_load"].mean() * 1.1
opt_max_demand = total_load_df["total_load"].max()
max_demand_lift_cost = (opt_max_demand - ori_max_demand) * 38.4
gross_income = origin_balance - opt_balance - max_demand_lift_cost

print("调度后最大需量：", opt_max_demand,
      "原始最大需量：", ori_max_demand,
      "需量抬升成本：", max_demand_lift_cost,
      "总收益：", gross_income)

调度后最大需量： 12713.087 原始最大需量： 10640.612925188974 需量抬升成本： 79583.00447274337 总收益： 354784.03433600755


In [65]:
total_load_df

,total_load,demand_load,es_load
2025-09-01 00:00:00,12097.087,9896.0,-2201.087
2025-09-01 00:05:00,12105.087,9904.0,-2201.087
2025-09-01 00:10:00,12097.087,9896.0,-2201.087
2025-09-01 00:15:00,12001.087,9800.0,-2201.087
2025-09-01 00:20:00,12081.087,9880.0,-2201.087
...,...,...,...
2025-09-30 23:35:00,11538.425,10280.0,-1258.425
2025-09-30 23:40:00,11550.357,10292.0,-1258.357
2025-09-30 23:45:00,11522.295,10264.0,-1258.295
2025-09-30 23:50:00,11510.241,10252.0,-1258.241
